In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# WORK STARTED ON UNIQUE DATA

In [2]:
work_files = {
    1: "J_1_Unique.csv",
    2: "J_2_Unique.csv",
    3: "J_3_Unique.csv",
    5: "J_5_Unique.csv"
}

In [3]:
def friction_model_unique(v, Tc, Bv, Vs):
    return (Tc * np.tanh(v / Vs) + Bv * v)

In [4]:
results = []

for joint, filename in work_files.items():
    print("\n" + "=" * 70)
    print(f"JOINT {joint}")
    print("="*70)

    # ==========================================
    # LOAD CSV
    # ==========================================
    file_path = os.path.join(folder_path, filename)
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    print("Columns:")
    print(df.columns.tolist())

    # ==========================================
    # CREATE COMMON COLUMNS
    # ==========================================
    df["v"] = np.deg2rad(df[f"v{joint}"])
    df["q"] = np.deg2rad(df[f"q{joint}"])
    df["tm"] = df[f"m_t_{joint}"]
    df["tj"] = df[f"jts{joint}"]
    df["tf"] = (N * df["tm"] - (-df["tj"]))

    # ==========================================
    # FEATURES & TARGET
    # ==========================================
    X = df[["v"]].values
    y = df["tf"].values

    # ==========================================
    # TRAIN = 70%
    # TEMP  = 30%
    # ==========================================
    (X_train, X_temp, y_train, y_temp) = train_test_split(X, y, test_size=0.30, random_state=42, shuffle=True)

    # ==========================================
    # VAL = 10%
    # TEST = 20%
    # ==========================================
    (X_val, X_test, y_val, y_test) = train_test_split(X_temp, y_temp, test_size=2/3, random_state=42, shuffle=True)
    print()
    print("Train :", X_train.shape)
    print("Val :", X_val.shape)
    print("Test :", X_test.shape)

    # ==========================================
    # FIT MODEL
    # ==========================================
    p0 = [5, 0.1, 0.01]
    params, _ = curve_fit(
        friction_model_unique,
        X_train.ravel(),
        y_train,
        p0=p0,
        maxfev=50000
    )

    Tc, Bv, Vs = params

    print("\nEstimated Parameters")
    print("Tc =", Tc)
    print("Bv =", Bv)
    print("Vs =", Vs)

    # ==========================================
    # VALIDATION
    # ==========================================
    val_pred = friction_model_unique(X_val.ravel(), *params)
    val_r2 = r2_score(y_val, val_pred)
    print(f"\nValidation R² = {val_r2:.4f}")

    # ==========================================
    # TEST
    # ==========================================

    pred = friction_model_unique(X_test.ravel(), *params)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    print("\nPerformance")
    print(f"RMSE = {rmse:.4f}")
    print(f"MAE = {mae:.4f}")
    print(f"R² = {r2:.4f}")
    results.append([joint, Tc, Bv, Vs, rmse, mae, r2])

    # ==========================================
    # PLOT 1
    # Actual vs Predicted
    # ==========================================
    plt.figure(figsize=(7,7))
    plt.scatter(y_test, pred, s=8, alpha=0.3)
    mn = min(y_test.min(), pred.min())
    mx = max(y_test.max(), pred.max())
    plt.plot([mn,mx], [mn,mx], 'r')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"Joint {joint}\nR²={r2:.4f}")
    plt.grid(True)
    plt.show()

    # ==========================================
    # PLOT 2
    # Residual
    # ==========================================
    residual = (y_test - pred)
    plt.figure(figsize=(8,4))
    plt.scatter(pred, residual, s=5, alpha=0.3)
    plt.axhline(0, color="red")
    plt.xlabel("Predicted")
    plt.ylabel("Residual")
    plt.title(f"Joint {joint} Residual Plot")
    plt.grid(True)
    plt.show()

    # ==========================================
    # PLOT 3
    # Friction Curve
    # ==========================================
    idx = np.argsort( X_test.ravel())
    plt.figure(figsize=(10,6))
    plt.scatter(X_test.ravel(), y_test, s=5, alpha=0.2, label="Measured")
    plt.plot(X_test.ravel()[idx], pred[idx], linewidth=3, label="Model")
    plt.xlabel("Velocity (rad/s)")
    plt.ylabel("Friction Torque")
    plt.title(f"Joint {joint}")
    plt.legend()
    plt.grid(True)
    plt.show()


JOINT 1


NameError: name 'folder_path' is not defined